# tool-call-agent: fine-tuning a small LLM to emit tool calls

Built during an internship at a telecom company. The question: can a small open model, fine-tuned on a free Colab GPU, turn a plain request like *"Top up line 9 with 50 bucks"* into JSON a Java backend can execute?

```json
{"tool": "chargeLine", "parameters": {"lineId": "9", "amount": 50.0}}
```

**What's in this notebook**

1. Install dependencies and check the GPU
2. Load Gemma 4 E2B (4-bit) and attach LoRA adapters
3. Generate a small synthetic dataset
4. Tokenize and fine-tune
5. Run inference and parse the JSON
6. Evaluate on 30 hand-written test prompts

**Before you run it**

- Use a **T4 GPU** runtime (Runtime > Change runtime type).
- Run the install cell, then **restart the session**, then run everything from the top of section 2.
- All data is synthetic and generated below. Nothing here comes from real customers or company systems.

**Known limits:** 50 training examples, one phrasing, one tool (`chargeLine`). The evaluation in section 6 has not been run yet. This is a prototype, not a finished agent.

## 1. Setup

### Install dependencies

Unsloth patches the model for faster, lower-memory fine-tuning. The `trl` pin is the version that worked for this run. An earlier version of this cell also pinned `xformers<0.0.30`, but that failed to build on current Colab, and Unsloth runs without xformers, so the pin was dropped.

After this cell finishes, restart the session (Runtime > Restart session, or Ctrl+M .) so Python picks up the new packages.

> **Note:** on my original run the `unsloth_zoo` import failed and I patched a few missing functions into the package by hand. I removed that hack from this version. If you hit `ImportError`s from `unsloth_zoo` on a newer or older release, reinstalling both `unsloth` and `unsloth-zoo` from GitHub with `--no-cache-dir` was what fixed it for me.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.15.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-uzv263g2/unsloth_0e26ec77fbc94509b00e9fc9f76310a9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-uzv263g2/unsloth_0e26ec77fbc94509b00e9fc9f76310a9
  Resolved https://github.com/unslothai/unsloth.git to commit 60248e78de459badb7a0e016195d2b806795370a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 32.4 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 104.2 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.7 MB/s  0:00:00
  Cr

### Check the GPU

Colab's free tier gives a Tesla T4 (about 15 GB). The T4 has no bfloat16 support, which is why the training arguments below set fp16. On the 2026-11-27 run, Unsloth reported that fp16 does not work for Gemma 4 and ran it in float32 instead.

In [2]:
!nvidia-smi

Fri Nov 27 13:07:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Load the model and add LoRA adapters

**Base model:** `unsloth/gemma-4-E2B-it-unsloth-bnb-4bit`, Gemma 4 E2B instruction tuned, quantized to 4 bits so it fits comfortably on a T4.

**LoRA** trains small adapter matrices instead of all the model weights. Here: rank 16, alpha 16, no dropout, applied to the attention (`q/k/v/o_proj`) and MLP (`gate/up/down_proj`) layers. `use_gradient_checkpointing="unsloth"` trades a little speed for much lower memory use.

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Gemma 4 E2B loaded with LoRA adapters")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.11.12: Fast Gemma4 patching. Transformers: 5.18.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Unsloth: the model has bfloat16 layers on GPU 0 (Tesla T4), which has no native bfloat16, so they run slowly. Load with dtype = torch.float16, or keep the model on bfloat16 GPUs with CUDA_VISIBLE_DEVICES.
Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Gemma 4 E2B loaded with LoRA adapters


## 3. Synthetic training data

The agent is meant to call Java methods. Three tools are defined in the project:

| Tool | Parameters |
|---|---|
| `chargeLine` | `lineId`, `amount` |
| `checkLineStatus` | `lineId` |
| `emergencyStop` | `lineId` |

Only `chargeLine` appears in the training data below, so the model has not been trained on the other two.

Each example is 50 generated requests of the form *"Charge {amt} to line {id}"*. The target output is a one-line reasoning step, then the JSON action:

```
<|think|>
Action: chargeLine. ID: 12. Amt: 50.
<|channel|>{"tool": "chargeLine", "parameters": {"lineId": "12", "amount": 50.0}}
```

The prompt tokens (`<|turn|>`, `<|think|>`, `<|channel|>`) are written by hand here, so they may not match Gemma's official chat template exactly. Training and inference use the same hand-written format, which is what matters for this experiment.

In [4]:
import json
import random
from datasets import Dataset

random.seed(3407)  # added for reproducibility; the original run was unseeded

dataset_samples = []
for _ in range(50):
    line_id = str(random.randint(1, 15))
    amount = random.choice([10, 20, 50, 100])
    dataset_samples.append({
        "content": f"Charge {amount} to line {line_id}",
        "thought": f"Action: chargeLine. ID: {line_id}. Amt: {amount}.",
        "action": {"tool": "chargeLine",
                   "parameters": {"lineId": line_id, "amount": float(amount)}},
    })

def format_chat(samples):
    return [{"text": (
        f"<bos><|turn|>system\nAgent.<|turn|>user\n{s['content']}"
        f"<|turn|>assistant\n<|think|>\n{s['thought']}\n"
        f"<|channel|>{json.dumps(s['action'])}<eos>"
    )} for s in samples]

dataset = Dataset.from_list(format_chat(dataset_samples))
print(dataset[0]["text"])

<bos><|turn|>system
Agent.<|turn|>user
Charge 100 to line 1<|turn|>assistant
<|think|>
Action: chargeLine. ID: 1. Amt: 100.
<|channel|>{"tool": "chargeLine", "parameters": {"lineId": "1", "amount": 100.0}}<eos>


## 4. Tokenize and train

**Tokenizing:** examples are padded or truncated to 128 tokens. The pad token is set to the end-of-sequence token. Unsloth wraps the tokenizer, so `tokenizer_only` pulls out the underlying Hugging Face tokenizer.

**Training:** the plain Hugging Face `Trainer`, not Unsloth's trainer, because the Unsloth chat-template route failed for me on this setup.

| Setting | Value |
|---|---|
| Batch size | 1, with 8 gradient accumulation steps (effective batch of 8) |
| Steps | 60 (about 10 passes over 50 examples) |
| Learning rate | 2e-4 |
| Optimizer | `adamw_8bit` |
| Precision | `fp16=True` in the arguments, but Unsloth ran Gemma 4 in float32 |

Watch the loss in the logs. On a dataset this small it should fall quickly.

In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

tokenizer_only = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
tokenizer_only.pad_token = tokenizer_only.eos_token

def tokenize(element):
    return tokenizer_only(element["text"], truncation=True,
                          max_length=128, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True,
                                remove_columns=dataset.column_names)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [6]:
trainer = Trainer(
    model = model,
    train_dataset = tokenized_dataset,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_only, mlm=False),
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = True,
        optim = "adamw_8bit",
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        output_dir = "tool_call_agent_v1",
        report_to = "none",
        logging_steps = 1,
    ),
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 24,158,208 of 5,128,455,712 (0.47% trained)


Step,Training Loss
1,4.578407
2,4.682499
3,3.817893
4,3.087699
5,2.519763
6,2.023611
7,1.568313
8,1.233985
9,0.889523
10,0.638399


Unsloth: Restored added_tokens_decoder metadata in tool_call_agent_v1/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.47730172984302044, metrics={'train_runtime': 361.7817, 'train_samples_per_second': 1.327, 'train_steps_per_second': 0.166, 'total_flos': 788628993343488.0, 'train_loss': 0.47730172984302044, 'epoch': 8.64})

## 5. Inference

Free-form prompting gave rambling output from this model, not clean JSON. I went through several fixes while debugging:

1. Greedy decoding plus a stop string
2. Priming the answer with `Action:`
3. A raw `USER: ... AGENT_ACTION:` format with a repetition penalty
4. Pre-filling the response up to the opening `{` of the JSON
5. Adding a format hint and a one-shot example to the system prompt

The version below combines the last two. It pre-fills `Action: chargeLine.` and the opening brace, so the model only has to produce the line ID and amount, then stops at the closing `}`.

**Limitation:** because the tool name is pre-filled, the model is not choosing the tool. Removing that pre-fill is the main next step (see the README).

In [7]:
FastLanguageModel.for_inference(model)

SPEC_HINT = 'Format: {"tool": "chargeLine", "parameters": {"lineId": "ID", "amount": VAL}}'

def tool_call_agent(user_input):
    prompt = (
        f"<bos><|turn|>system\nAgent. {SPEC_HINT}<|turn|>user\n{user_input}"
        f"<|turn|>assistant\n<|think|>\nAction: chargeLine.\n<|channel|>{{"
    )
    inputs = tokenizer_only(prompt, return_tensors="pt",
                            add_special_tokens=False).to("cuda")
    outputs = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = 64,
        do_sample = False,          # greedy decoding
        stop_strings = ["}"],       # stop when the arguments object closes
        tokenizer = tokenizer_only,
    )
    generated = tokenizer_only.batch_decode(outputs)[0][len(prompt):]
    # the prompt already ends with "{" and the model often repeats it, so keep exactly one
    raw = "{" + generated.lstrip().lstrip("{")
    raw = raw[:raw.rfind("}") + 1]  # drop anything after the last brace
    # the stop string fires on the first "}", which closes "parameters",
    # so add back any closing braces that are missing
    return raw + "}" * (raw.count("{") - raw.count("}"))

### Try it

Edit the prompts to test your own. The first one is phrased differently from anything in the training data, so it's a quick check of how well the model handles new wording.

In [8]:
test_prompts = [
    "Top up line 9 with 50 bucks.",
    "Add 250 units to line 12 immediately.",
    "Charge 20 to line 3",
]

for p in test_prompts:
    print(f"INPUT:  {p}")
    print(f"OUTPUT: {tool_call_agent(p)}\n")

INPUT:  Top up line 9 with 50 bucks.
OUTPUT: {"tool": "chargeLine", "parameters": {"lineId": "9", "amount": 50.0}}

INPUT:  Add 250 units to line 12 immediately.
OUTPUT: {"tool": "chargeLine", "parameters": {"lineId": "12", "amount": 250.0}}

INPUT:  Charge 20 to line 3
OUTPUT: {"tool": "chargeLine", "parameters": {"lineId": "3", "amount": 20.0}}



### Parse the output

The backend needs valid JSON, so the last step is to parse and sanity check the result before passing it on. Anything that fails here should be rejected, not executed.

In [9]:
def parse_action(raw):
    try:
        action = json.loads(raw)
        params = action["parameters"]
        assert action["tool"] == "chargeLine"
        assert isinstance(params["lineId"], str)
        assert float(params["amount"]) > 0
        return action
    except (json.JSONDecodeError, KeyError, AssertionError, ValueError, TypeError):
        return None

for p in test_prompts:
    print(p, "->", parse_action(tool_call_agent(p)))

Top up line 9 with 50 bucks. -> {'tool': 'chargeLine', 'parameters': {'lineId': '9', 'amount': 50.0}}
Add 250 units to line 12 immediately. -> {'tool': 'chargeLine', 'parameters': {'lineId': '12', 'amount': 250.0}}
Charge 20 to line 3 -> {'tool': 'chargeLine', 'parameters': {'lineId': '3', 'amount': 20.0}}


## What's next

- Train on all three tools, plus vague and invalid requests the agent should refuse or ask about.
- Run the evaluation in section 6 and report exact-match accuracy on tool name and arguments.
- Remove the pre-filled `Action: chargeLine.` so the model picks the tool itself.
- Use constrained decoding against a JSON schema.
- Save the LoRA adapter (`model.save_pretrained(...)`) or export a merged model, and call it from a mock Java service.

## 6. Evaluation

`data/test_prompts.jsonl` holds 30 hand-written `chargeLine` requests with the expected JSON for each. The wording is much more varied than the training data ("top up", "add credit", "put N on line X", "bucks", "euros", different word order), and some amounts (15, 75, 200, 250, 500) never appear in training.

For each prompt the cells below run `tool_call_agent`, then `parse_action`, and count:

- how many outputs are valid JSON
- how many match the expected tool, `lineId` and `amount` exactly

Failures are listed in a table at the end. The first cell clones the repo if the test file isn't already next to the notebook, which is the usual case in Colab.

In [10]:
import os
import subprocess

TEST_FILE = "data/test_prompts.jsonl"
if not os.path.exists(TEST_FILE):
    if not os.path.exists("tool-call-agent"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohammadhamayel23/tool-call-agent.git"], check=True)
    TEST_FILE = "tool-call-agent/data/test_prompts.jsonl"

with open(TEST_FILE, encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(test_cases)} test cases from {TEST_FILE}")

Cloning into 'tool-call-agent'...


Loaded 30 test cases from tool-call-agent/data/test_prompts.jsonl


In [11]:
import pandas as pd

rows = []
for case in test_cases:
    raw = tool_call_agent(case["prompt"])
    try:
        json.loads(raw)
        valid_json = True
    except json.JSONDecodeError:
        valid_json = False
    action = parse_action(raw)
    expected = case["expected"]
    match = (
        action is not None
        and action["tool"] == expected["tool"]
        and action["parameters"]["lineId"] == expected["parameters"]["lineId"]
        and float(action["parameters"]["amount"]) == expected["parameters"]["amount"]
    )
    rows.append({
        "prompt": case["prompt"],
        "expected_lineId": expected["parameters"]["lineId"],
        "expected_amount": expected["parameters"]["amount"],
        "output": raw,
        "valid_json": valid_json,
        "match": match,
    })

results = pd.DataFrame(rows)
n = len(results)
print(f"Valid JSON:  {results['valid_json'].sum()}/{n}")
print(f"Exact match: {results['match'].sum()}/{n} (tool, lineId and amount)")

Valid JSON:  30/30
Exact match: 30/30 (tool, lineId and amount)


In [12]:
failures = results[~results["match"]]
print(f"{len(failures)} failures")
pd.set_option("display.max_colwidth", None)
failures[["prompt", "expected_lineId", "expected_amount", "output", "valid_json"]]

0 failures


,prompt,expected_lineId,expected_amount,output,valid_json
